# Pancreatic Tumor Segmentation — Training Notebook
**Master Thesis | Sugimoto Lab | Biomedical Engineering**

Run this notebook on **Google Colab Pro** (A100 recommended).

---

## 1. Mount Google Drive & Set Up Environment

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
REPO = '/content/drive/MyDrive/tesi'
os.chdir(REPO)
print(f'Working directory: {os.getcwd()}')
print('Contents:', os.listdir('.'))

In [ ]:
# Install dependencies (only needed on first run, cached afterward)
!pip install monai[all] nibabel scipy scikit-image pyyaml tensorboard --quiet

In [ ]:
import sys
sys.path.insert(0, REPO)

import torch
print(f'PyTorch: {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

## 2. Choose Experiment
Set `EXPERIMENT` to one of: `supervised` | `semisup` | `selfsup_pretrain` | `selfsup_finetune` | `fewshot`

In [ ]:
EXPERIMENT   = 'semisup'    # ← change this
OUTPUT_DIR   = f'/content/drive/MyDrive/tesi/experiments/{EXPERIMENT}'
CONFIG_PATH  = f'/content/drive/MyDrive/tesi/configs/{EXPERIMENT if EXPERIMENT != "selfsup_finetune" else "selfsup_pretrain"}.yaml'
RESUME_FROM  = None   # set to checkpoint path to resume, e.g. OUTPUT_DIR + '/checkpoint_epoch100.pth'

print(f'Experiment:  {EXPERIMENT}')
print(f'Config:      {CONFIG_PATH}')
print(f'Output dir:  {OUTPUT_DIR}')

## 3. Load Config & Build Data Loaders

In [ ]:
from src.utils.io import load_config

cfg = load_config(CONFIG_PATH)

# Override batch_size and num_workers for Colab
cfg['training']['batch_size'] = 2
cfg['training']['num_workers'] = 2

print('Config loaded:')
import yaml; print(yaml.dump(cfg, default_flow_style=False))

In [ ]:
data_cfg   = cfg['data']
patch_size = tuple(data_cfg.get('patch_size', [96, 96, 96]))

if EXPERIMENT == 'supervised':
    from src.data.dataset import PancreasDataset
    train_ds = PancreasDataset(data_cfg['data_dir'], split='train',
                                val_fold=data_cfg['val_fold'], patch_size=patch_size)
    val_ds   = PancreasDataset(data_cfg['data_dir'], split='val',
                                val_fold=data_cfg['val_fold'])
    train_loader = train_ds.get_loader(batch_size=cfg['training']['batch_size'],
                                        num_workers=cfg['training']['num_workers'])
    val_loader   = val_ds.get_loader(batch_size=1, shuffle=False,
                                      num_workers=cfg['training']['num_workers'])
    print(f'Train: {len(train_ds.dataset)} cases | Val: {len(val_ds.dataset)} cases')

elif EXPERIMENT in ('semisup',):
    from src.data.dataset import SemiSupervisedDataset
    semisup_cfg = cfg['semisup']
    ds = SemiSupervisedDataset(
        data_dir      = data_cfg['data_dir'],
        labeled_ratio = semisup_cfg['labeled_ratio'],
        val_fold      = data_cfg['val_fold'],
        unlabeled_dir = data_cfg.get('unlabeled_dir', None),
        patch_size    = patch_size,
    )
    labeled_loader   = ds.get_labeled_loader(batch_size=cfg['training']['batch_size'],
                                              num_workers=cfg['training']['num_workers'])
    unlabeled_loader = ds.get_unlabeled_loader(batch_size=cfg['training']['batch_size'],
                                                num_workers=cfg['training']['num_workers'])
    val_loader       = ds.get_val_loader(num_workers=cfg['training']['num_workers'])

elif EXPERIMENT in ('selfsup_pretrain',):
    from src.data.dataset import PancreasDataset
    from monai.data import Dataset as MonaiDataset
    from src.data.transforms import get_unlabeled_transforms
    from pathlib import Path
    unlabeled_dir = data_cfg.get('unlabeled_dir', data_cfg['data_dir'] + '/imagesTr')
    imgs = sorted(Path(unlabeled_dir).glob('*.nii.gz'))
    data = [{'image': str(p)} for p in imgs]
    unlabeled_ds = MonaiDataset(data=data, transform=get_unlabeled_transforms(patch_size))
    from torch.utils.data import DataLoader
    unlabeled_loader = DataLoader(unlabeled_ds, batch_size=cfg['training']['batch_size'],
                                   shuffle=True, num_workers=cfg['training']['num_workers'])
    print(f'Unlabeled CT volumes: {len(data)}')

elif EXPERIMENT == 'fewshot':
    from src.data.dataset import FewShotEpisodeDataset
    from torch.utils.data import DataLoader
    fs_cfg = cfg['fewshot']
    train_ds = FewShotEpisodeDataset(
        data_dir=data_cfg['data_dir'], n_shot=fs_cfg['n_shot'],
        n_query=fs_cfg['n_query'], n_episodes=fs_cfg['n_episodes'],
        val_fold=data_cfg['val_fold'], split='train')
    val_ds = FewShotEpisodeDataset(
        data_dir=data_cfg['data_dir'], n_shot=fs_cfg['n_shot'],
        n_query=fs_cfg['n_query'], n_episodes=fs_cfg['n_val_eps'],
        val_fold=data_cfg['val_fold'], split='val')
    train_loader = DataLoader(train_ds, batch_size=1, shuffle=True,
                               num_workers=cfg['training']['num_workers'])
    val_loader   = DataLoader(val_ds, batch_size=1, shuffle=False,
                               num_workers=cfg['training']['num_workers'])

print('Data loaders ready')

## 4. Build Trainer & Train

In [ ]:
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

if EXPERIMENT == 'supervised':
    from src.training.trainer_supervised import SupervisedTrainer
    trainer = SupervisedTrainer(cfg, OUTPUT_DIR, device=DEVICE, resume=RESUME_FROM)
    best = trainer.train(train_loader, val_loader)

elif EXPERIMENT == 'semisup':
    from src.training.trainer_semisup import SemiSupervisedTrainer
    trainer = SemiSupervisedTrainer(cfg, OUTPUT_DIR, device=DEVICE, resume=RESUME_FROM)
    best = trainer.train(labeled_loader, unlabeled_loader, val_loader)

elif EXPERIMENT == 'selfsup_pretrain':
    from src.training.trainer_selfsup import SelfSupervisedPretrainer
    trainer = SelfSupervisedPretrainer(cfg, OUTPUT_DIR, device=DEVICE)
    best = trainer.train(unlabeled_loader)

elif EXPERIMENT == 'selfsup_finetune':
    ENCODER_CKPT = f'/content/drive/MyDrive/tesi/experiments/selfsup_pretrain/encoder_pretrained.pth'
    from src.training.trainer_selfsup import SelfSupFineTuner
    from src.data.dataset import PancreasDataset
    train_ds = PancreasDataset(data_cfg['data_dir'], split='train', val_fold=data_cfg['val_fold'])
    val_ds   = PancreasDataset(data_cfg['data_dir'], split='val',   val_fold=data_cfg['val_fold'])
    train_loader = train_ds.get_loader(batch_size=2, num_workers=2)
    val_loader   = val_ds.get_loader(batch_size=1, shuffle=False, num_workers=2)
    trainer = SelfSupFineTuner(cfg, ENCODER_CKPT, OUTPUT_DIR, device=DEVICE)
    best = trainer.train(train_loader, val_loader)

elif EXPERIMENT == 'fewshot':
    from src.training.trainer_fewshot import FewShotTrainer
    trainer = FewShotTrainer(cfg, OUTPUT_DIR, device=DEVICE, resume=RESUME_FROM)
    best = trainer.train(train_loader, val_loader)

print(f'\nBest checkpoint: {best}')

## 5. TensorBoard Monitoring

In [ ]:
%load_ext tensorboard
%tensorboard --logdir /content/drive/MyDrive/tesi/experiments